# Task adapter and decoding latency (Colab, GPU)

Two experiments requested in review. **A. Task adapter**: one LoRA adapter per model trained on the PubMedQA question-answering task itself (`scripts/train_task_adapter.py`: the harness prompt of the task `pubmedqa` followed by the answer word, loss on the answer tokens only; 10,000 yes/no examples of `pqa_artificial` plus the training half of the 1,000 expert-labeled questions, split deterministically by pubid order and numpy seed 0), then evaluated along lambda for logit-space and weight-space scaling on the other half of the labeled questions (harness task `pubmedqa_test500`, 500 questions with maybe, `scripts/zeroshot_bench.py --tasks pubmedqa_test500`), on the four general zero-shot benchmarks, and on general text for the KL divergence that the equal-divergence comparison of the paper uses (`scripts/likelihood_tradeoff.py --sets general`). **B. Latency**: milliseconds per generated token and peak memory of the base model, the plain adapter, weight-space scaling and logit-space scaling (two cached forward passes per step) for a batch of 20 prompts (`scripts/latency.py`), with the PubMed adapters of the paper already on Drive.

Results go to `/content/drive/MyDrive/dynalora_results/task` (`adapters/`, `pubmedqa/`, `bench/`, `likelihood/`, `analysis/`) and `/content/drive/MyDrive/dynalora_results/latency`, with per-question scores only; every step skips the outputs that already exist, so the notebook can be re-run after a disconnection. The datasets are loaded from the Hugging Face Hub at run time and nothing of them is written to Drive.

Runtime: T4 for GPT-2 (section 1, about 45 min in all: training about 15-25 min, PubMedQA about 5 min, general benchmarks about 15-20 min, KL about 2 min); T4 or L4 for Qwen2.5-1.5B (section 2, about 2-3 h on a T4, roughly half on an L4: training about 1-1.5 h, PubMedQA about 10 min, benchmarks about 30 min, KL about 5 min); the latency section takes a few minutes per model. After switching the runtime, run the two setup cells again.

In [ ]:
!nvidia-smi -L
!git clone -b paper https://github.com/alexroehrs/dynalora.git /content/dynalora
%cd /content/dynalora
!git pull -q
!pip install -q -r requirements.txt lm-eval bitsandbytes
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
for d in ['task/adapters', 'task/pubmedqa', 'task/bench', 'task/likelihood', 'task/analysis', 'latency']:
    os.makedirs('/content/drive/MyDrive/dynalora_results/' + d, exist_ok=True)
print(os.listdir('/content/drive/MyDrive/dynalora_results/task'))

## 1. GPT-2 (T4)

### 1.1 Train the task adapter

One epoch over the 10,500 examples with an effective batch of 16 (4 x 4), max length 1,024 tokens (the context length of GPT-2; longer examples are skipped and counted in `train_log.txt`), gradient checkpointing, fp16 base model. The log reports the validation loss and the accuracy of the harness scoring on 200 held-out training examples before and after training.

In [ ]:
!python scripts/train_task_adapter.py --model gpt2 --batch_size 4 --grad_accum 4 --adapters_dir /content/drive/MyDrive/dynalora_results/task/adapters
!head -n 1 /content/drive/MyDrive/dynalora_results/task/adapters/gpt2__pubmedqa_task/train_log.txt

### 1.2 PubMedQA test half along lambda (both forms, the 13 default lambdas)

In [ ]:
!python scripts/zeroshot_bench.py --model gpt2 --adapters_dir /content/drive/MyDrive/dynalora_results/task/adapters --configs pubmedqa_task --tasks pubmedqa_test500 --max_len 1024 --batch_size 8 --out /content/drive/MyDrive/dynalora_results/task/pubmedqa

### 1.3 General zero-shot benchmarks (LAMBADA, HellaSwag, PIQA, ARC-Easy, first 1,000 documents)

In [ ]:
!python scripts/zeroshot_bench.py --model gpt2 --adapters_dir /content/drive/MyDrive/dynalora_results/task/adapters --configs pubmedqa_task --limit 1000 --out /content/drive/MyDrive/dynalora_results/task/bench

### 1.4 KL divergence from the base model on general text (WikiText-103, as in the paper)

In [ ]:
!python scripts/likelihood_tradeoff.py --model gpt2 --sets general --adapters_dir /content/drive/MyDrive/dynalora_results/task/adapters --configs pubmedqa_task --out /content/drive/MyDrive/dynalora_results/task/likelihood

## 2. Qwen2.5-1.5B (T4 or L4)

Same steps; effective batch 16 as 2 x 8, lambda 0, 0.5, ..., 3 and the first 500 documents of each general benchmark, as for the other Qwen2.5-1.5B runs of the paper.

In [ ]:
!python scripts/train_task_adapter.py --model Qwen/Qwen2.5-1.5B --batch_size 2 --grad_accum 8 --adapters_dir /content/drive/MyDrive/dynalora_results/task/adapters
!head -n 1 /content/drive/MyDrive/dynalora_results/task/adapters/Qwen2.5-1.5B__pubmedqa_task/train_log.txt

In [ ]:
!python scripts/zeroshot_bench.py --model Qwen/Qwen2.5-1.5B --adapters_dir /content/drive/MyDrive/dynalora_results/task/adapters --configs pubmedqa_task --tasks pubmedqa_test500 --max_len 1024 --batch_size 4 --lambdas 0 0.5 1 1.5 2 2.5 3 --out /content/drive/MyDrive/dynalora_results/task/pubmedqa

In [ ]:
!python scripts/zeroshot_bench.py --model Qwen/Qwen2.5-1.5B --adapters_dir /content/drive/MyDrive/dynalora_results/task/adapters --configs pubmedqa_task --batch_size 8 --limit 500 --lambdas 0 0.5 1 1.5 2 2.5 3 --out /content/drive/MyDrive/dynalora_results/task/bench

In [ ]:
!python scripts/likelihood_tradeoff.py --model Qwen/Qwen2.5-1.5B --sets general --adapters_dir /content/drive/MyDrive/dynalora_results/task/adapters --configs pubmedqa_task --out /content/drive/MyDrive/dynalora_results/task/likelihood

## 3. Optional: Qwen2.5-7B in 4 bits (L4)

The base model is loaded in 4-bit NF4 for training (QLoRA) and in every evaluation, as in `notebooks/colab_qwen7b.ipynb`. Uncomment to run (training takes several hours on an L4).

In [ ]:
# !python scripts/train_task_adapter.py --model Qwen/Qwen2.5-7B --load_in_4bit --batch_size 1 --grad_accum 16 --adapters_dir /content/drive/MyDrive/dynalora_results/task/adapters
# !python scripts/zeroshot_bench.py --model Qwen/Qwen2.5-7B --adapters_dir /content/drive/MyDrive/dynalora_results/task/adapters --configs pubmedqa_task --tasks pubmedqa_test500 --load_in_4bit --max_len 1024 --batch_size 4 --lambdas 0 0.5 1 1.5 2 2.5 3 --out /content/drive/MyDrive/dynalora_results/task/pubmedqa
# !python scripts/zeroshot_bench.py --model Qwen/Qwen2.5-7B --adapters_dir /content/drive/MyDrive/dynalora_results/task/adapters --configs pubmedqa_task --load_in_4bit --batch_size 4 --limit 500 --lambdas 0 0.5 1 1.5 2 2.5 3 --out /content/drive/MyDrive/dynalora_results/task/bench
# !python scripts/likelihood_tradeoff.py --model Qwen/Qwen2.5-7B --sets general --adapters_dir /content/drive/MyDrive/dynalora_results/task/adapters --configs pubmedqa_task --load_in_4bit --batch_size 4 --out /content/drive/MyDrive/dynalora_results/task/likelihood

## 4. Decoding latency of the two forms (T4)

For each model, 50 new tokens for a batch of 20 prompts under the base model, the plain adapter (weight-space lambda 1), weight-space scaling at lambda 2 and logit-space scaling at lambda 2 (two cached forward passes per step), one warm-up and 5 timed repetitions with a fixed sampling seed; `latency__<model>.csv` reports the median milliseconds per generated token per sequence, its interquartile range and the peak GPU memory. The PubMed adapters of the paper (`/content/drive/MyDrive/dynalora_results/gpt2/adapters`, `/content/drive/MyDrive/dynalora_results/qwen15/adapters`) are used; the task adapters of this notebook give the same timings (commented lines).

In [ ]:
!python scripts/latency.py --model gpt2 --adapters_dir /content/drive/MyDrive/dynalora_results/gpt2/adapters --config pubmed --out /content/drive/MyDrive/dynalora_results/latency
!python scripts/latency.py --model Qwen/Qwen2.5-1.5B --adapters_dir /content/drive/MyDrive/dynalora_results/qwen15/adapters --config pubmed --out /content/drive/MyDrive/dynalora_results/latency
# !python scripts/latency.py --model gpt2 --adapters_dir /content/drive/MyDrive/dynalora_results/task/adapters --config pubmedqa_task --out /content/drive/MyDrive/dynalora_results/latency
# !python scripts/latency.py --model Qwen/Qwen2.5-1.5B --adapters_dir /content/drive/MyDrive/dynalora_results/task/adapters --config pubmedqa_task --out /content/drive/MyDrive/dynalora_results/latency

## 5. Quick look: accuracy along lambda, by gold answer, general benchmarks, and at equal KL divergence

The same analysis as `scripts/analysis/task_adapter.py` in the repository (`task_adapter_curves.csv`, `task_adapter_by_answer.csv`, `task_adapter_eq.csv`), written next to the results.

In [ ]:
!python scripts/analysis/task_adapter.py --inputs /content/drive/MyDrive/dynalora_results/task/pubmedqa --bench /content/drive/MyDrive/dynalora_results/task/bench --likelihood /content/drive/MyDrive/dynalora_results/task/likelihood --out /content/drive/MyDrive/dynalora_results/task/analysis --by_answer

## 6. Package the results for download (scores and timings only, no questions, abstracts or adapter weights)

In [ ]:
!cd /content/drive/MyDrive/dynalora_results && zip -qr /content/dynalora_task.zip task latency -x "task/adapters/*"
from google.colab import files
files.download('/content/dynalora_task.zip')